In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
from fba_tertiary_design_io import (
    assert_environ_settings,
    get_fn,
    read_yaml,
    assert_tertiary_settings,
    get_tile_centers_grid,
    get_tile_centers_rosette,
    create_tiles_table,
    creates_priority_table,
    finalize_target_table,
    assert_files,
    create_targets_assign,
    plot_targets_assign,
subsample_targets_avail,
create_empty_priority_dict,
)

from desimodel.focalplane.geometry import get_tile_radius_deg

from astropy.table import Table
from astropy import wcs
import astropy.coordinates as coord
from astropy.visualization.wcsaxes import SphericalCircle
import astropy.units as u
import fitsio
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic


params = {
    "legend.fontsize": "x-large",
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
    "figure.facecolor": "w",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif"
}
plt.rcParams.update(params)

In [ ]:
targets = Table(fitsio.read("/global/cfs/cdirs/desi/users/bid13/DESI_II/TEST8_COSMOS_LSSTY1_target_list.fits"))


In [ ]:
fba = Table(fitsio.read("./fba_output/tertiary-targets-9999-assign.fits"))

fba = fba[fba["NASSIGN"]>0]

In [ ]:
_ = plt.hist(fba["I_MAG"],bins=100,label="observed")
_ = plt.hist(targets["I_MAG"],bins=100,histtype="step",label="parent")
plt.xlabel(r"$i$-mag")
plt.ylabel("Counts")
plt.yscale("log")

In [ ]:
# base_path = Path(os.environ["CFS"]) / "desi" / "users" / "bid13" / "DESI_II"

# spec_cat = Table.read( base_path / "other_surveys" / "COSMOS2020" / "COSMOS2020_FARMER_R1_v2.2_p3.fits").to_pandas()
# mask = (spec_cat['FLAG_COMBINED']==0) & ((spec_cat['lp_type']==0) | (spec_cat['lp_type']==2))
# spec_cat = spec_cat[mask]
# z_list = np.zeros(len(spec_cat))
# z_list[(spec_cat['lp_type']==0).to_numpy(dtype=bool)] = spec_cat["lp_zBEST"][(spec_cat['lp_type']==0)]
# z_list[(spec_cat['lp_type']==2).to_numpy(dtype=bool)] = spec_cat["lp_zq"][(spec_cat['lp_type']==2)]
# spec_cat["Z"] = z_list



In [ ]:
def redshift_dessert(z):
    mask = z<=1.65
    return mask.sum()

In [ ]:
# fracs, bins, _ = binned_statistic(spec_cat["HSC_i_MAG"], spec_cat["Z"], statistic=redshift_dessert, bins=np.linspace(18,24.5,21), range=(18,24.5))
# count, bin_edges, _ = binned_statistic(spec_cat["HSC_i_MAG"], spec_cat["Z"], bins=np.linspace(18,24.5,21), range=(18,24.5),statistic="count")



# from statsmodels.stats.proportion import proportion_confint
# ci_lo, ci_upp = proportion_confint(fracs,count,method="beta")

# fracs /=count

In [ ]:
bin_edges = np.linspace(18,25,21)
bins = (bin_edges[1:] + bin_edges[:-1])/2

In [ ]:
from scipy.integrate import trapezoid
def z0_for_limit(maglimit, band='i'):
    if band=='i':
        # z0 = -0.724785  +  0.0408182 * maglimit #21.5 < i < 23 # numbers from Jeff's email
        z0 = -0.744  +  0.0417 * maglimit #Number from DESC SRD 
    elif band=='r':
        z0 = -1.10356 +    0.0567708 * maglimit #for 22 < r < 24
    return z0
def dNdz(z, z0, alpha ):
    return z**2 * np.exp((-z/z0)**alpha)/(2*z0**3)
def dNdz_cumulative_mag(z_grid,maglimit,alpha=1,band='i'):
    z0 = z0_for_limit(maglimit, band=band)
    return dNdz(z_grid, z0, alpha=alpha)
def N_total(i_lim):
    # return 42.9*10**(0.359*(i_lim-25)) # Number from DESC SRD
    return 46*10**(0.31*(i_lim-25)) #Number from LSST Science Book
threshold=1.6
frac_greater_list =[]
for i in range(len(bin_edges)-1):
    z_grid = np.linspace(threshold,100,5000)
    numerator = N_total(bin_edges[i+1])*trapezoid(dNdz_cumulative_mag(z_grid,bin_edges[i+1]),z_grid) - N_total(bin_edges[i])*trapezoid(dNdz_cumulative_mag(z_grid,bin_edges[i]),z_grid)
    denominator = N_total(bin_edges[i+1])*trapezoid(dNdz_cumulative_mag(np.linspace(0,100,5000),bin_edges[i+1]),np.linspace(0,100,5000)) - N_total(bin_edges[i])*trapezoid(dNdz_cumulative_mag(np.linspace(0,100,5000),bin_edges[i]),np.linspace(0,100,5000))
    frac_greater_list.append(numerator/denominator)

In [ ]:
fig, ax = plt.subplots(1,1,)
ax.scatter(fba["I_MAG"],fba["SUCCESS_PROB"],s=1,alpha=0.5,rasterized=True,label="Observed Targets")

# ax.plot(bins,fracs,ls="--",color="k")
ax.plot(bins,1-np.array(frac_greater_list),ls="--",color="k",label=r"$z\leq1.6$ Fraction")
# ax.fill_between(bins, ci_lo, ci_upp, alpha=0.1, color="k")
ax.set_ylim(0.66,1.01)
ax.set_xlim(17.5,25)
ax.axhline(0.80,ls="--",color="k",alpha=0.3)
ax.axvline(24.1,ls="--",color="k",alpha=0.3)
# ax.axvline(23.5,ls="--",color="k",alpha=0.3)
# ax.axhline(0.90,ls="--",color="k",alpha=0.3)
ax.set_xlabel(r"$i$-mag")
ax.set_ylabel(r"Predicted Success Rate")
ax.legend(frameon=False,markerscale=3)
plt.savefig("predicted_success.pdf",bbox_inches="tight")

In [ ]:
def sigmoid(y):
    return 1/(1+np.exp(-y))
    
def success_prob_old(magnitude, exptime, A=-1.1970 , B=28.7410):
    mag_transformed = magnitude - 1.25*np.log10(exptime/6000)
    success = sigmoid(A*mag_transformed + B)
    return success

def success_prob(magnitude, exptime, const=27.6624 , x1=-1.1504,x2=1.7981):
    
    success = sigmoid(const + x1*magnitude + x2*np.log10(exptime/6000))
    return success

In [ ]:
fig, ax = plt.subplots(1,1,)

s_prob = success_prob_old(fba["I_MAG"], fba["EXP_TIME"])
ax.scatter(fba["I_MAG"],s_prob,s=1,alpha=0.5,rasterized=True,label="Observed Targets")
ax.plot(bins,fracs,ls="--",color="k",)
ax.set_ylim(0.67,1.01)
ax.set_xlim(17.5,25)
ax.axhline(0.80,ls="--",color="k",alpha=0.3)
ax.axvline(24.1,ls="--",color="k",alpha=0.3)
ax.axvline(23.5,ls="--",color="k",alpha=0.3)
ax.axhline(0.90,ls="--",color="k",alpha=0.3)
ax.set_xlabel(r"$i$-mag")
ax.set_ylabel(r"Predicted Success Rate")
# plt.savefig("predicted_success.pdf",bbox_inches="tight")

In [ ]:
def creates_priority_table(yamlfn="./fba_output/tertiary-config-9999.yaml", no_stop_targets={"LSST_Y1_P1":7500}):

    # AR read the requested settings
    mydict = read_yaml(yamlfn)["samples"]
    # AR initiate a dictionary
    myd = create_empty_priority_dict()
    # # AR loop over target samples
    # if no_stop_targets is not None:
    #     filterByKey = lambda keys: {x: mydict[x] for x in keys}
    #     no_stop_targets = filterByKey(no_stop_targets)
    #     for t in no_stop_targets:
    #         del mydict[t]
    #     for target in no_stop_targets:
    #         prio_init, ngoal = no_stop_targets[target]["PRIORITY_INIT"],94# no_stop_targets[target]["NGOAL"]
    #         for nmin in range(ngoal + 1):
    #             nmax, prio = nmin, prio_init + nmin
    #             myd["TERTIARY_TARGET"].append(target)
    #             myd["NUMOBS_DONE_MIN"].append(nmin)
    #             myd["NUMOBS_DONE_MAX"].append(nmax)
    #             myd["PRIORITY"].append(prio)
                

    for target in mydict:
        prio_init, ngoal = mydict[target]["PRIORITY_INIT"], mydict[target]["NGOAL"]
        for nmin in range(ngoal + 1):
            if nmin == ngoal:
                if target in no_stop_targets:
                    nmax, prio = 99, no_stop_targets[target]
                else:
                    nmax, prio = 99, 1
            else:
                nmax, prio = nmin, prio_init + nmin
            myd["TERTIARY_TARGET"].append(target)
            myd["NUMOBS_DONE_MIN"].append(nmin)
            myd["NUMOBS_DONE_MAX"].append(nmax)
            myd["PRIORITY"].append(prio)
    # AR convert to Table()
    d = Table()
    for key in myd:
        d[key] = myd[key]
    return d

In [ ]:
creates_priority_table().write("test_priorities.csv",overwrite=True)

In [ ]:
target_ra = targets["RA"]
target_dec = targets["DEC"]

In [ ]:
plt.figure(figsize=(10,10),)
ax = plt.subplot( )
ax.scatter(target_ra,target_dec,marker=".",alpha=0.1) #can also use 'icrs' 'galactic' or any other astropy.coordinates object.
ax.scatter(tile_ras,tile_decs,marker=".")
ax.set_xlabel("RA")
ax.set_ylabel("DEC")


for tile_ra, tile_dec in zip(tile_ras,tile_decs):
    circle = SphericalCircle((tile_ra*u.degree,tile_dec*u.degree),tile_rad*u.degree)
    ax.plot(circle.get_xy()[:,0],circle.get_xy()[:,1],c="C1")

In [ ]:
plot_targets_assign(9999, "/pscratch/sd/b/bid13/photo_z_spare_fiber/desi_deep/fiber_assign/fba_output")

In [ ]:
ta = Table(fitsio.read("./fba_output/tertiary-targets-9999-assign.fits"))

In [ ]:
ta = ta[ta["NASSIGN"]>0]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3.33,3))
ax.scatter(ta["I_MAG"],ta["SUCCESS_PROB"],s=1,alpha=0.5,rasterized=True)
ax.set_ylim(0.6,1.01)
ax.set_xlim(17.5,25)
ax.axhline(0.80,ls="--",color="k",alpha=0.3)
ax.axvline(24.1,ls="--",color="k",alpha=0.3)
ax.axvline(23.5,ls="--",color="k",alpha=0.3)
ax.axhline(0.90,ls="--",color="k",alpha=0.3)
ax.set_xlabel(r"$i$-mag")
ax.set_ylabel(r"Predicted Success Rate")
# plt.savefig("predicted_success.pdf",bbox_inches="tight")

In [ ]:
def sigmoid(y):
    return 1/(1+np.exp(-y))
    
def success_prob(magnitude, exptime, A=-1.1701, B=28.1211):
    mag_transformed = magnitude - 1.25*np.log10(exptime/6000)
    success = sigmoid(A*mag_transformed + B)
    return success

In [ ]:
goaltime = 1000

ta["EXP_TIME"] = ta["NASSIGN"]*goaltime

ta["SUCCESS_PROB"] = success_prob(ta["I_MAG"],ta["EXP_TIME"])
ta

In [ ]:
yamlfn = "./fba_output/tertiary-config-9999.yaml"

mydict = read_yaml(yamlfn)


In [ ]:
# def create_targets(yamlfn, outfn):
#     mydict = read_yaml(yamlfn)
#     d = Table(fitsio.read(mydict["settings"]["target_list_fn"]))
#     d["TERTIARY_TARGET"] = np.zeros(len(d), dtype=object)
    
#     for key, value in mydict["samples"].items():
#         mask = (d["I_MAG"]<value["I_MAG_MAX"]) & (d["I_MAG"]>=value["I_MAG_MIN"])
#         d["TERTIARY_TARGET"][mask] = key
        
    
#     # AR finalize
#     d = finalize_target_table(d, yamlfn)
#     d.meta["RANDSEED"] = read_yaml(yamlfn)["settings"]["np_rand_seed"]
#     d.write(outfn)

In [ ]:
target = Table(fitsio.read("COSMOS_LSSTY1_target_list.fits"))

In [ ]:
d = Table(fitsio.read(mydict["settings"]["target_list_fn"]))
d["TERTIARY_TARGET"] = np.zeros(len(d), dtype=object)


In [ ]:
for key, value in mydict["samples"].items():
    mask = (d["I_MAG"]<value["I_MAG_MAX"]) & (d["I_MAG"]>=value["I_MAG_MIN"])
    d["TERTIARY_TARGET"][mask] = key
d["TERTIARY_TARGET"] = d["TERTIARY_TARGET"].astype(str)

In [ ]:
d = subsample_targets_avail(
        d,
        mydict["settings"]["prognum"],
        mydict["settings"]["targdir"],
        mydict["settings"]["rundate"],
        ignore_samples="LSST_Y1_P4",
    )

In [ ]:
d = finalize_target_table(d, yamlfn)

In [ ]:
d.meta["RANDSEED"] = read_yaml(yamlfn)["settings"]["np_rand_seed"]
d.write("/pscratch/sd/b/bid13/photo_z_spare_fiber/desi_deep/fiber_assign/fba_output/tertiary-targets-9999.fits",overwrite=True)

In [ ]:
np.unique(d["TERTIARY_TARGET"])

In [ ]:
plt.hist(d["I_MAG"],bins=100)

In [ ]:
for i in np.unique(d["TERTIARY_TARGET"]):
    print(f"{i}:{len(d[d['TERTIARY_TARGET']==i])}")


In [ ]:
def solve_quadratic(success, A, B, C):
    odds = logit(success)
    return (-B - np.sqrt(B**2 - 4*A*(C-odds)))/(2*A)

def solve_linear(success, A, B):
    odds = logit(success)
    return (odds - B)/A

def time_to_success(success, magnitude, A, B, C=None):
    if C is not None:
        magnitude_transformed = solve_quadratic(success, A=A, B=B, C=C)
        # print(magnitde_transformed)
    else:
        magnitude_transformed = solve_linear(success, A=A, B=B)
    return 100*10**((magnitude - magnitude_transformed)/1.25) #minutes

def time_to_survey(success, magnitude, multiplexing, total_number, A, B, C=None, hours_per_night=8):
    # Fixed Mirror size: Account for that
    
    time_per_spectra = time_to_success(success, magnitude, A=A, B=B, C=C)
    time_required_minutes = time_per_spectra*total_number/multiplexing
    return time_required_minutes/(60*hours_per_night) #yrs


def pred_success(magnitude,exptime, multiplexing, total_number, A, B, A_err, B_err, res, hours_per_night=8):
    mag_transformed = magnitude - 1.25*np.log10(exptime*60/100)
    success = sigmoid(A*mag_transformed + B)
    y_pred = res.get_prediction(poly.fit_transform(mag_transformed[:,None]))
    # success_int = (sigmoid(A_err[0]*mag_transformed + B_err[0]), sigmoid(A_err[1]*mag_transformed + B_err[1]))
    return y_pred.predicted_mean, y_pred.conf_int()
    # return success, success_int

# solve_quadratic(0.95, A=0.0477, B=-3.3865, C=53.8232)
# solve_linear(0.95, A=-1.1572, B=27.8182)
# print(""time_to_success(0.85, 22.5, A=-1.1572, B=27.8182))
print(f"Total time required: {time_to_survey(0.85, 24.1, 4500, 30000, A=res.params[1], B=res.params[0], hours_per_night=6):.2f} nights (Linear Model)")
# print(f"Total time required: {time_to_survey(0.90, 24.1, 4000, 30000, A=0.0477, B=-3.3865, C=53.8232):.2f} nights (Quadratic Model)")

In [ ]:
mydict = read_yaml("./fba_output/tertiary-config-9999.yaml")

In [ ]:
no_stop_targets = ["LSST_Y1_P1", "LSST_Y1_P2"]




filterByKey = lambda keys: {x: mydict[x] for x in keys}


no_stop_targets = filterByKey(no_stop_targets)

for t in no_stop_targets:
    del mydict[t]


In [ ]:
no_stop_targets

In [ ]:
mydict

In [ ]:
cat = Table(fitsio.read(path))

In [ ]:
cat["NASSIGN"]